# Mental Health Text Triage: from BERT to a Quantized DistilBERT

Classifying short social-media style text into seven mental-health categories
(Normal, Depression, Suicidal, Anxiety, Stress, Bipolar, Personality disorder),
using the Kaggle *Sentiment Analysis for Mental Health* corpus.

The notebook works through four things:

1. **Fine-tune BERT** as a reference model.
2. **Audit the corpus** for duplicates, label conflicts, and train/test leakage.
   This turned out to matter more than expected.
3. **Optimize for deployment** along four axes: the classification head, the
   encoder, the inference runtime, and numeric precision. Two of the four don't work.
4. **Evaluate honestly** with per-class metrics, calibration, bootstrap confidence
   intervals, and an edge-case suite.

The model that ends up deployed is DistilBERT quantized to int8 and served through
ONNX Runtime: about 4x faster and 6x smaller than the BERT reference on CPU, for
roughly 0.8 points of accuracy.

## Setup

Run the install cell once, then restart the kernel. Python 3.11 or 3.12 --> the ML
stack doesn't have wheels for anything newer yet. If you don't have an NVIDIA GPU,
swap `cu128` for `cpu` in the torch install (fine-tuning will be slow, but every
evaluation cell here runs on CPU anyway).

`kagglehub` needs a Kaggle API token. Get one from kaggle.com/settings -> API ->
Create New Token, and drop the `kaggle.json` into `~/.kaggle/` (or
`%USERPROFILE%\\.kaggle\\` on Windows). Once per machine.

In [ ]:
# ONE-TIME INSTALL: run once, then restart the kernel. Skip on later runs.

# PyTorch with CUDA. Use the cu128 wheels; older cu121/cu124 builds silently fall
# back to CPU on newer cards. No GPU? Replace cu128 with cpu.
%pip install "torch>=2.7.0" --index-url https://download.pytorch.org/whl/cu128

# Everything else, pinned to a mutually compatible set. transformers<5 is what keeps
# optimum working.
%pip install "transformers==4.44.2" "optimum[onnxruntime]==1.22.0" "accelerate==0.34.2" "datasets==2.21.0" "evaluate==0.4.3"
%pip install "onnxruntime==1.19.2" "scikit-learn==1.5.2" "numpy==1.26.4" "pandas==2.2.2" seaborn matplotlib kagglehub gdown ipywidgets streamlit onnxscript

print("\nInstall complete: restart the kernel before continuing.")

In [ ]:
import sys
assert (3, 9) <= sys.version_info < (3, 13), (
    f"This notebook targets Python 3.9-3.12, but you're on {sys.version.split()[0]}. "
    "Newer versions don't have wheels yet for torch / onnxruntime / optimum."
)

import torch, transformers, datasets, sklearn
print("torch       :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

## Data

53,043 rows of raw text with a `status` label. Drop the nulls, label-encode the
seven classes, and make a stratified 80/20 split. Everything downstream reuses
this exact split (`random_state=42`), so results stay comparable.

In [ ]:
import kagglehub
suchintikasarkar_sentiment_analysis_for_mental_health_path = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')

print('Data source import complete.')

In [ ]:
import pandas as pd

df = pd.read_csv(suchintikasarkar_sentiment_analysis_for_mental_health_path+'/Combined Data.csv', index_col=0)

In [ ]:
df.dropna(inplace = True)

In [ ]:
from datasets import Dataset, ClassLabel, Features, Value
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode labels for the entire dataset first to ensure consistent mapping
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['status'])

# Create id2label and label2id mappings
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

# Split the dataset into train and test again with the new 'labels' column
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['labels'], random_state=42)

# Convert pandas DataFrames to Hugging Face Dataset objects
# Ensure 'statement' and 'labels' are present (corrected from 'original_statement')
train_dataset = Dataset.from_pandas(train_df[['statement', 'labels']], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['statement', 'labels']], preserve_index=False)

# Add feature information to the datasets (optional but good practice)
train_dataset = train_dataset.cast_column('labels', ClassLabel(num_classes=len(label_encoder.classes_), names=label_encoder.classes_.tolist()))
test_dataset = test_dataset.cast_column('labels', ClassLabel(num_classes=len(label_encoder.classes_), names=label_encoder.classes_.tolist()))

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Label mappings: id2label = {id2label}, label2id = {label2id}")

## Corpus audit: duplicates, label conflicts, and leakage

The Kaggle corpus is an aggregation of nine separately scraped datasets, and nobody
deduplicated across them. That's worth checking before trusting any accuracy number,
because duplicate rows can land on both sides of the train/test split.

Three things get measured here: how much of the corpus is duplicated, how often an
identical statement carries two different labels, and how many test rows also appear
in training. The last one is the one that inflates results.

The cell is standalone: it rebuilds the split itself and saves a mask of the leaked
rows so later cells can score on a clean subset.

In [ ]:
# CORPUS AUDIT: duplication, label conflicts, train/test leakage
# Standalone. Needs only phase1_1_preds.npz (for part 3). No model loading.
import re, os, gc, warnings
import numpy as np, pandas as pd
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# Audit parameters (report these verbatim in the paper)
SEED          = 42
TEST_SIZE     = 0.20
COS_THRESHOLD = 0.90     # char n-gram cosine above which a pair is a near-duplicate
JACCARD_MIN   = 0.50     # word-overlap floor confirming a near-duplicate
MIN_LEN       = 30       # normalized characters required to confirm a near-duplicate
NGRAM         = (3, 5)   # character n-gram range
MAX_FEATURES  = 300_000
MIN_DF        = 2
CHUNK         = 128      # test rows per similarity block
SWEEP         = [0.80, 0.85, 0.90, 0.95, 0.99]
PREDS_NPZ     = "phase1_1_preds.npz"

def normalize(s):
    """Lowercase, strip non-alphanumerics, collapse whitespace."""
    s = re.sub(r"[^a-z0-9\s]", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

# Load and split exactly as everywhere else
import kagglehub
p = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
raw = pd.read_csv(p + '/Combined Data.csv', index_col=0)
n_raw = len(raw)
df = raw.dropna().copy()
n_null = n_raw - len(df)
le = LabelEncoder(); df['labels'] = le.fit_transform(df['status'])
CLASSES = list(le.classes_)
train_df, test_df = train_test_split(df, test_size=TEST_SIZE,
                                     stratify=df['labels'], random_state=SEED)

print("=" * 84); print("PART 1: DUPLICATION AND LABEL VALIDITY"); print("=" * 84)
print(f"raw rows                {n_raw:,}")
print(f"rows dropped (null)     {n_null:,}")
print(f"usable rows             {len(df):,}")
print(f"train / test            {len(train_df):,} / {len(test_df):,}")

df['norm'] = [normalize(s) for s in df['statement']]
n_unique = df['norm'].nunique()
dup_rows = len(df) - n_unique
counts = df['norm'].value_counts()
repeated = counts[counts > 1]
print(f"\nunique normalized statements  {n_unique:,}")
print(f"duplicate rows                {dup_rows:,} ({100*dup_rows/len(df):.2f}%)")
print(f"statements that repeat        {len(repeated):,}")
print(f"normalized length < 10 chars  {(df['norm'].str.len() < 10).sum():,}")
print(f"normalized to empty string    {(df['norm'].str.len() == 0).sum():,}")

# label conflicts among repeated statements
grp = df[df['norm'].isin(repeated.index)].groupby('norm')['status']
conflicts = []
for norm, s in grp:
    u = s.unique()
    if len(u) > 1:
        conflicts.append({"statement": norm[:80], "n_rows": len(s),
                          "n_labels": len(u),
                          "labels": ", ".join(f"{k}({v})" for k, v
                                              in Counter(s).most_common())})
conf_df = pd.DataFrame(conflicts).sort_values("n_rows", ascending=False)
n_conf_rows = int(conf_df["n_rows"].sum()) if len(conf_df) else 0
print(f"\nrepeated statements with conflicting labels  "
      f"{len(conf_df):,} of {len(repeated):,} "
      f"({100*len(conf_df)/max(len(repeated),1):.2f}%)")
print(f"rows affected                                {n_conf_rows:,}")
print("\nLargest label conflicts:")
print(conf_df.head(12).to_string(index=False))
conf_df.to_csv("phase2_D_label_conflicts.csv", index=False)

print("\nMost frequent repeated statements (content check):")
top = counts.head(12).rename_axis("statement").reset_index(name="count")
top["statement"] = top["statement"].str.slice(0, 60)
print(top.to_string(index=False))

# Part 2: leakage
print("\n" + "=" * 84); print("PART 2: TRAIN/TEST LEAKAGE"); print("=" * 84)
tr_norm = np.array([normalize(s) for s in train_df['statement']])
te_norm = np.array([normalize(s) for s in test_df ['statement']])
te_labels = test_df['labels'].values

tr_set = set(tr_norm.tolist())
exact = np.array([t in tr_set for t in te_norm])
print(f"exact normalized matches   {exact.sum():,} of {len(te_norm):,} "
      f"({100*exact.mean():.2f}%)")

print(f"\nfitting char {NGRAM[0]}-{NGRAM[1]}-gram TF-IDF "
      f"(max_features={MAX_FEATURES:,}, min_df={MIN_DF})...")
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=NGRAM,
                      max_features=MAX_FEATURES, min_df=MIN_DF, lowercase=False)
Xtr = vec.fit_transform(tr_norm)      # already L2-normalized by TfidfVectorizer
Xte = vec.transform(te_norm)
print(f"vocabulary {len(vec.vocabulary_):,}   "
      f"nnz train {Xtr.nnz:,}  test {Xte.nnz:,}")

XtrT = Xtr.T.tocsc()
max_cos = np.zeros(Xte.shape[0], dtype=np.float32)
arg_tr  = np.zeros(Xte.shape[0], dtype=np.int64)
for i in tqdm(range(0, Xte.shape[0], CHUNK), desc="similarity search"):
    block = (Xte[i:i + CHUNK] @ XtrT).toarray()
    max_cos[i:i + CHUNK] = block.max(axis=1)
    arg_tr [i:i + CHUNK] = block.argmax(axis=1)
    del block
gc.collect()

def jaccard(a, b):
    A, B = set(a.split()), set(b.split())
    return len(A & B) / len(A | B) if (A or B) else 0.0

near_raw = (max_cos >= COS_THRESHOLD) & (~exact)
confirmed = np.zeros_like(near_raw)
for i in np.where(near_raw)[0]:
    t, m = te_norm[i], tr_norm[arg_tr[i]]
    if len(t) >= MIN_LEN and jaccard(t, m) >= JACCARD_MIN:
        confirmed[i] = True

loose  = exact | near_raw
strict = exact | confirmed
n_discarded = int(near_raw.sum() - confirmed.sum())

print(f"\nnear-duplicate candidates (cos >= {COS_THRESHOLD})  {int(near_raw.sum()):,}")
print(f"  confirmed by word overlap                     {int(confirmed.sum()):,}")
print(f"  discarded as short-text collisions            {n_discarded:,}")
print(f"\nSTRICT leaked test rows  {int(strict.sum()):,} "
      f"({100*strict.mean():.2f}%)")
print(f"LOOSE  leaked test rows  {int(loose.sum()):,} "
      f"({100*loose.mean():.2f}%)")

print("\nExamples discarded by the word-overlap filter:")
shown = 0
for i in np.where(near_raw & ~confirmed)[0]:
    print(f"  cos={max_cos[i]:.3f}  jac={jaccard(te_norm[i], tr_norm[arg_tr[i]]):.2f}"
          f"  test={te_norm[i][:44]!r}  train={tr_norm[arg_tr[i]][:44]!r}")
    shown += 1
    if shown >= 6: break

print("\nThreshold sensitivity (strict rule at each cosine cutoff):")
sweep_rows = []
for th in SWEEP:
    cand = (max_cos >= th) & (~exact)
    conf = np.zeros_like(cand)
    for i in np.where(cand)[0]:
        if len(te_norm[i]) >= MIN_LEN and \
           jaccard(te_norm[i], tr_norm[arg_tr[i]]) >= JACCARD_MIN:
            conf[i] = True
    m = exact | conf
    sweep_rows.append({"cosine_threshold": th, "leaked_rows": int(m.sum()),
                       "leaked_pct": round(100 * m.mean(), 2)})
    print(f"  {th:.2f} -> {int(m.sum()):,} ({100*m.mean():.2f}%)")
pd.DataFrame(sweep_rows).to_csv("phase2_D_threshold_sweep.csv", index=False)

print("\nPer-class leakage rate (strict):")
pc = []
for i, c in enumerate(CLASSES):
    m = te_labels == i
    pc.append({"class": c, "test_rows": int(m.sum()),
               "leaked": int(strict[m].sum()),
               "leaked_pct": round(100 * strict[m].mean(), 2)})
pc_df = pd.DataFrame(pc).sort_values("leaked_pct", ascending=False)
print(pc_df.to_string(index=False))
pc_df.to_csv("phase2_D_perclass_leakage.csv", index=False)

np.savez_compressed("phase2_D_leakage_mask.npz",
                    strict=strict, loose=loose, exact=exact,
                    max_cos=max_cos, arg_train=arg_tr, labels=te_labels)
print("\nSaved phase2_D_leakage_mask.npz")

# Part 3: re-score every variant on the clean subset
print("\n" + "=" * 84)
print("PART 3: EFFECT ON REPORTED PERFORMANCE")
print("=" * 84)
if not os.path.exists(PREDS_NPZ):
    print(f"{PREDS_NPZ} not found: run cell 1.1 first to generate it.")
else:
    z = np.load(PREDS_NPZ)
    saved_labels = z["labels"]
    if len(saved_labels) != len(te_labels) or not np.array_equal(saved_labels, te_labels):
        print("WARNING: labels in the predictions file do not match this split.\n"
              "         Confirm cell 1.1 used the same SEED and TEST_SIZE.")
    clean = ~strict
    rows = []
    for k in [f for f in z.files if f != "labels"]:
        yp = z[k]
        for name, m in [("full", np.ones_like(strict)),
                        ("leaked", strict), ("clean", clean)]:
            rows.append({
                "Model": k.replace("_", " "), "subset": name, "n": int(m.sum()),
                "Acc": round(accuracy_score(te_labels[m], yp[m]), 4),
                "MacroF1": round(f1_score(te_labels[m], yp[m], average="macro",
                                          zero_division=0), 4)})
    res = pd.DataFrame(rows)
    wide = res.pivot(index="Model", columns="subset", values=["Acc", "MacroF1"])
    print(wide.to_string())
    print("\nInflation from leakage (full minus clean):")
    for k in [f for f in z.files if f != "labels"]:
        nm = k.replace("_", " ")
        r = res[res.Model == nm].set_index("subset")
        print(f"  {nm:28} accuracy {r.loc['full','Acc']-r.loc['clean','Acc']:+.4f}   "
              f"macro-F1 {r.loc['full','MacroF1']-r.loc['clean','MacroF1']:+.4f}")
    res.to_csv("phase2_D_clean_metrics.csv", index=False)
    print("\nWrote phase2_D_clean_metrics.csv")
    print("\nLeakage inflates macro-F1 more than accuracy because duplication")
    print("concentrates in the smallest classes, which macro-averaging weights equally.")

## Fine-tuning BERT

The reference configuration: `bert-base-uncased` with its native classification head,
3 epochs, learning rate 2e-5, batch size 16, weight decay 0.01, FP16 mixed precision.
Sequences are truncated to 128 tokens, which is what the deployed app uses.

If you don't want to wait for training, the download cell below pulls an
already-fine-tuned checkpoint instead.

In [ ]:
import gdown
import zipfile
import os
import shutil

# Download the model zip from Google Drive
file_id = '1pkEdzzrVLL8pgVrzr3h7-hR73D72co6n'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'bert_sentiment_model.zip'
final_model_dir = './bert_sentiment_model'

if not os.path.exists(final_model_dir):
    gdown.download(url, output, quiet=False)
    # Extract to a temporary directory to identify the folder name inside the zip
    temp_extract = './temp_extract'
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall(temp_extract)

    # Identify the extracted folder and rename/move it to bert_sentiment_model
    extracted_contents = os.listdir(temp_extract)
    if extracted_contents:
        source_path = os.path.join(temp_extract, extracted_contents[0])
        shutil.move(source_path, final_model_dir)

    # Cleanup
    shutil.rmtree(temp_extract)
    if os.path.exists(output): os.remove(output)
    print(f"Model downloaded and folder renamed to {final_model_dir}")

#model_name = "bert-base-uncased"
model_name = "./bert_sentiment_model"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["statement"], truncation=True, padding=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"Tokenized training dataset features: {tokenized_train_dataset.column_names}")
print(f"Tokenized testing dataset features: {tokenized_test_dataset.column_names}")

In [ ]:
import numpy as np
import evaluate

# Load the pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id)

# Load the accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(), # Mixed precision only where a GPU supports it
    report_to="none",              # don't attempt wandb/tensorboard logging locally
)
print("Training arguments defined.")

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("Trainer initialized.")

Training run (slow). Skip it if you downloaded the checkpoint above.

In [ ]:
# YOU SHOULD ONLY NEED TO DO THIS ONCE
trainer.train()
print("Model training complete.")

### Evaluating the fine-tuned model

In [ ]:
results = trainer.evaluate()
print("Model evaluation complete.")
print(results)

# Generate predictions for the test set to create a confusion matrix
predictions = trainer.predict(tokenized_test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = tokenized_test_dataset["labels"]

# Compute confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=id2label.values(),
            yticklabels=id2label.values())
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - BERT Sentiment Analysis')
plt.show()

## Optimization axis 1: the classification head

The first idea was to use BERT purely as a feature extractor and put a Random Forest
on the 768-dimensional `[CLS]` embeddings, on the theory that a lightweight classifier
would speed up inference.

It doesn't, and the reason is structural: every input still has to pass through the
full encoder to produce that embedding, and the encoder forward pass is where
essentially all of the time goes. Swapping a single 768-to-7 linear layer for 300
decision trees adds work rather than removing it.

Keeping this here because the negative result is the thing that pointed at the encoder.

In [ ]:
import torch
from torch.utils.data import DataLoader
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

# Ensure the model is on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def get_bert_embeddings(dataset, model, batch_size=32):
    model.eval() # Set model to evaluation mode
    embeddings = []
    labels = []

    # Filter the dataset to include only the columns needed by the model and labels
    # This prevents the data_collator from trying to process the 'statement' column
    dataset_for_dataloader = dataset.remove_columns(['statement'])

    # The data_collator defined earlier helps in padding and preparing batches
    # num_workers=0 avoids multiprocessing DataLoader hangs in Jupyter on Windows
    dataloader = DataLoader(dataset_for_dataloader, batch_size=batch_size, collate_fn=data_collator, num_workers=0, pin_memory=torch.cuda.is_available())

    with torch.no_grad(): # Disable gradient calculations for inference
        for batch in tq.tqdm(dataloader, desc="Extracting Embeddings"):
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)

            # Extract labels from the batch
            batch_labels = batch['labels'].cpu().numpy()

            # Get the base BERT model output (last hidden state)
            # AutoModelForSequenceClassification has a 'bert' attribute for the base model
            outputs = model.bert(input_ids=input_ids, attention_mask=attention_mask)

            # The [CLS] token embedding is at index 0 of the last_hidden_state
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

            embeddings.append(cls_embeddings)
            labels.append(batch_labels)

    return np.vstack(embeddings), np.concatenate(labels)

In [ ]:
from tqdm.auto import tqdm
import tqdm as tq

# Extract embeddings for train and test datasets
X_train_bert_embeddings, y_train_bert = get_bert_embeddings(tokenized_train_dataset, model)
X_test_bert_embeddings, y_test_bert = get_bert_embeddings(tokenized_test_dataset, model)

print(f"Shape of extracted BERT train embeddings: {X_train_bert_embeddings.shape}")
print(f"Shape of extracted BERT test embeddings: {X_test_bert_embeddings.shape}")

In [ ]:
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. TRAINING RANDOM FOREST (WITH PROGRESS TRACKING)
print("="*40)
print("Training Optimized Random Forest...")
print("="*40)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    class_weight='balanced',
    random_state=101,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train_bert_embeddings, y_train_bert)
y_pred_rf = rf_model.predict(X_test_bert_embeddings)

# 2. EVALUATION & VISUALIZATION
print("\n" + "="*40)
print("Random Forest Evaluation Metrics:")
print("="*40)
print(f"Overall Accuracy: {accuracy_score(y_test_bert, y_pred_rf):.4f}\n")

print("Classification Report:")
# Uses label_encoder explicitly to prevent NameErrors
print(classification_report(y_test_bert, y_pred_rf, target_names=label_encoder.classes_))

# Generate and plot Confusion Matrix
conf_matrix_rf = confusion_matrix(y_test_bert, y_pred_rf)
plt.figure(figsize=(10, 8))
sns.heatmap(
    conf_matrix_rf,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix for Random Forest (Optimized BERT Embeddings)')
plt.tight_layout()
plt.show()

# 3. EXPORT TRAINED MODEL
file_name = 'random_forest_bert_model.pkl'
with open(file_name, 'wb') as file:
    pickle.dump(rf_model, file)

print(f"\n[SUCCESS] Optimized Random Forest model saved to: {file_name}")

### Does fine-tuning actually matter?

Second question on the same axis: could we skip fine-tuning entirely and just use
embeddings from an off-the-shelf encoder? That would save the training cost.

The cell below runs three configurations under identical settings (fine-tuned encoder
with its native head, fine-tuned encoder plus Random Forest, and frozen encoder plus
Random Forest) and scores all of them on the leakage-filtered test set.

Between them these two experiments say the encoder is both where the cost is and where
the capability is, so the only lever left is a *smaller* encoder.

In [ ]:
# ABLATIONS AT CANONICAL CONFIG (MAX_LEN=128, same split, clean + full subsets)
# Standalone. Needs ./bert_sentiment_model and phase2_D_leakage_mask.npz
import os, time, warnings, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             classification_report)
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

MAX_LEN, BATCH, SEED, THREADS = 128, 32, 42, 2
RF_TREES, RF_DEPTH = 300, 25
FINETUNED, PRETRAINED = "./bert_sentiment_model", "bert-base-uncased"
MASK_NPZ = "phase2_D_leakage_mask.npz"
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"embedding device: {DEV}  (accuracy is device-independent; "
      f"no latency from this cell enters the efficiency tables)")

import kagglehub
p = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
df = pd.read_csv(p + '/Combined Data.csv', index_col=0).dropna()
le = LabelEncoder(); df['labels'] = le.fit_transform(df['status'])
CLASSES = list(le.classes_); S = CLASSES.index("Suicidal")
tr, te = train_test_split(df, test_size=0.2, stratify=df['labels'],
                          random_state=SEED)
Xtr_t = tr['statement'].astype(str).tolist(); ytr = tr['labels'].values
Xte_t = te['statement'].astype(str).tolist(); yte = te['labels'].values
clean = ~np.load(MASK_NPZ)["strict"]
print(f"train {len(ytr):,} | test {len(yte):,} | clean {int(clean.sum()):,}")

tok = AutoTokenizer.from_pretrained(FINETUNED)

def cls_embeddings(model_path, texts, desc):
    m = AutoModel.from_pretrained(model_path).to(DEV).eval()
    out = []
    for i in tqdm(range(0, len(texts), BATCH), desc=desc, leave=False):
        enc = tok(texts[i:i+BATCH], return_tensors="pt", truncation=True,
                  padding=True, max_length=MAX_LEN).to(DEV)
        with torch.no_grad():
            out.append(m(**enc).last_hidden_state[:, 0].cpu().numpy())
    del m; torch.cuda.empty_cache() if DEV == "cuda" else None
    return np.concatenate(out)

def score(name, y_pred):
    r = {"Configuration": name}
    for sub, msk in [("full", np.ones_like(clean)), ("clean", clean)]:
        r[f"Acc ({sub})"]     = round(accuracy_score(yte[msk], y_pred[msk]), 4)
        r[f"MacroF1 ({sub})"] = round(f1_score(yte[msk], y_pred[msk],
                                       average="macro", zero_division=0), 4)
        r[f"SuicR ({sub})"]   = round(recall_score(yte[msk], y_pred[msk],
                                       average=None, zero_division=0)[S], 4)
    return r

rows, preds = [], {}

# 1. reference: fine-tuned encoder + native classification head
clf = AutoModelForSequenceClassification.from_pretrained(FINETUNED).to(DEV).eval()
out = []
for i in tqdm(range(0, len(Xte_t), BATCH), desc="native head", leave=False):
    enc = tok(Xte_t[i:i+BATCH], return_tensors="pt", truncation=True,
              padding=True, max_length=MAX_LEN).to(DEV)
    with torch.no_grad():
        out.append(clf(**enc).logits.argmax(-1).cpu().numpy())
yp = np.concatenate(out); preds["native"] = yp
rows.append(score("Fine-tuned encoder + native head (reference)", yp))
W = clf.classifier.weight.detach().cpu().clone()
B = clf.classifier.bias.detach().cpu().clone()
del clf; torch.cuda.empty_cache() if DEV == "cuda" else None

# 2. head axis: fine-tuned encoder + Random Forest
Etr = cls_embeddings(FINETUNED, Xtr_t, "embed train (fine-tuned)")
Ete = cls_embeddings(FINETUNED, Xte_t, "embed test  (fine-tuned)")
rf_ft = RandomForestClassifier(n_estimators=RF_TREES, max_depth=RF_DEPTH,
                               class_weight="balanced", random_state=SEED,
                               n_jobs=-1).fit(Etr, ytr)
yp = rf_ft.predict(Ete); preds["ft_rf"] = yp
rows.append(score("Fine-tuned encoder + Random Forest", yp))

# 3. encoder axis: frozen (not fine-tuned) encoder + Random Forest
Ptr = cls_embeddings(PRETRAINED, Xtr_t, "embed train (pretrained)")
Pte = cls_embeddings(PRETRAINED, Xte_t, "embed test  (pretrained)")
rf_fz = RandomForestClassifier(n_estimators=RF_TREES, max_depth=RF_DEPTH,
                               class_weight="balanced", random_state=SEED,
                               n_jobs=-1).fit(Ptr, ytr)
yp = rf_fz.predict(Pte); preds["frozen_rf"] = yp
rows.append(score("Frozen encoder + Random Forest", yp))

res = pd.DataFrame(rows)
print("\n" + "=" * 96)
print(f"ABLATIONS: MAX_LEN={MAX_LEN}, seed={SEED}, RF({RF_TREES} trees, depth {RF_DEPTH})")
print("=" * 96)
print(res.to_string(index=False))

print("\nPer-class F1 on the clean subset:")
pc = {}
for k, nm in [("native", "native head"), ("ft_rf", "fine-tuned + RF"),
              ("frozen_rf", "frozen + RF")]:
    rep = classification_report(yte[clean], preds[k][clean], labels=range(7),
                                target_names=CLASSES, output_dict=True,
                                zero_division=0)
    pc[nm] = {c: round(rep[c]["f1-score"], 4) for c in CLASSES}
print(pd.DataFrame(pc).reindex(CLASSES).to_string())

# head-only cost, given a precomputed embedding (pinned CPU)
torch.set_num_threads(THREADS)
emb = Ete[:1000].astype(np.float32)
lin = torch.nn.Linear(768, 7); lin.weight.data, lin.bias.data = W, B; lin.eval()
with torch.no_grad():
    for i in range(50): lin(torch.from_numpy(emb[i:i+1]))
    t0 = time.perf_counter()
    for i in range(1000): lin(torch.from_numpy(emb[i:i+1]))
    lin_ms = (time.perf_counter() - t0) / 1000 * 1000
for i in range(50): rf_ft.predict(emb[i:i+1])
t0 = time.perf_counter()
for i in range(1000): rf_ft.predict(emb[i:i+1])
rf_ms = (time.perf_counter() - t0) / 1000 * 1000

print(f"\nHead-only cost per request, CPU @ {THREADS} threads "
      f"(embedding already computed):")
print(f"  linear head (768 -> 7)          {lin_ms:7.3f} ms")
print(f"  Random Forest ({RF_TREES} trees)      {rf_ms:7.3f} ms   "
      f"({rf_ms/lin_ms:.1f}x the linear head)")
print(f"  encoder forward pass            186.400 ms  (from the canonical benchmark)")
print(f"  -> head is {100*lin_ms/186.4:.2f}% of total with a linear head, "
      f"{100*rf_ms/(186.4+rf_ms):.2f}% with the Random Forest")

res.to_csv("ablations_canonical.csv", index=False)
np.savez("ablations_preds.npz", labels=yte, **preds)
print("\nWrote ablations_canonical.csv, ablations_preds.npz")

## Optimization axes 2-4: encoder, runtime, precision

DistilBERT is a distilled version of BERT with roughly 40% fewer parameters. Fine-tune
it the same way, then export both encoders to ONNX and quantize each to 8-bit integers.

Three separate changes, varied one at a time so the contribution of each is attributable.

In [ ]:
# Fine-tune DistilBERT
import numpy as np, evaluate, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)
from sklearn.metrics import classification_report

MAX_LEN = 128   # social posts are short; check test_df["statement"].str.len() to confirm
distil_ckpt = "distilbert-base-uncased"
distil_tokenizer = AutoTokenizer.from_pretrained(distil_ckpt)

# Re-tokenize from the RAW datasets: DistilBERT has no token_type_ids, so the
# BERT-tokenized datasets aren't reusable.
def _distil_tok(ex):
    return distil_tokenizer(ex["statement"], truncation=True, padding=True, max_length=MAX_LEN)
distil_train = train_dataset.map(_distil_tok, batched=True)
distil_test  = test_dataset.map(_distil_tok,  batched=True)

distil_model = AutoModelForSequenceClassification.from_pretrained(
    distil_ckpt, num_labels=len(id2label), id2label=id2label, label2id=label2id)

_metric = evaluate.load("accuracy")
def _compute(p):
    logits, labels = p
    return _metric.compute(predictions=np.argmax(logits, -1), references=labels)

distil_trainer = Trainer(
    model=distil_model,
    args=TrainingArguments(
        output_dir="./distil_results", eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="accuracy",
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=3, weight_decay=0.01, fp16=torch.cuda.is_available(), report_to="none"),
    train_dataset=distil_train, eval_dataset=distil_test,
    data_collator=DataCollatorWithPadding(tokenizer=distil_tokenizer),
    compute_metrics=_compute)
distil_trainer.train()

distil_dir = "./distilbert_sentiment_model"
distil_trainer.save_model(distil_dir); distil_tokenizer.save_pretrained(distil_dir)

d_pred = np.argmax(distil_trainer.predict(distil_test).predictions, -1)
print(classification_report(np.array(distil_test["labels"]), d_pred,
                            target_names=label_encoder.classes_, digits=4))

ONNX's newer dynamo exporter is flaky on this stack, so force the older TorchScript path first.

In [ ]:
import importlib, torch.onnx
importlib.reload(torch.onnx)          # reset to PyTorch's real export(), undoing any earlier patch
_original_export = torch.onnx.export
def _export_legacy(*args, **kwargs):
    kwargs["dynamo"] = False           # force the older, more stable TorchScript-based exporter
    return _original_export(*args, **kwargs)
torch.onnx.export = _export_legacy

In [ ]:
# ONNX export + dynamic int8 quantization (CPU)
import os
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from transformers import AutoTokenizer

# optimum's cleanup step sometimes tries to delete an external-data shard file
# (model.onnx.data) that was never created for models this size; harmless to skip.
_orig_remove = os.remove
def _safe_remove(path, *a, **kw):
    try:
        _orig_remove(path, *a, **kw)
    except FileNotFoundError:
        pass
os.remove = _safe_remove

def export_and_quantize(model_dir, out_prefix):
    onnx_fp32, onnx_int8 = f"{out_prefix}_onnx", f"{out_prefix}_onnx_int8"
    ort = ORTModelForSequenceClassification.from_pretrained(model_dir, export=True)
    ort.save_pretrained(onnx_fp32)
    AutoTokenizer.from_pretrained(model_dir).save_pretrained(onnx_fp32)
    quantizer = ORTQuantizer.from_pretrained(onnx_fp32)
    # avx2 runs on essentially any modern x86 CPU. If your CPU AND the deploy target
    # support AVX-512 VNNI, swap avx2 -> avx512_vnni for a faster quantized model.
    qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=onnx_int8, quantization_config=qconfig)
    AutoTokenizer.from_pretrained(model_dir).save_pretrained(onnx_int8)
    print(f"  {model_dir} -> {onnx_fp32}, {onnx_int8}")
    return onnx_fp32, onnx_int8

bert_onnx,   bert_onnx_int8   = export_and_quantize("./bert_sentiment_model",       "bert")
distil_onnx, distil_onnx_int8 = export_and_quantize("./distilbert_sentiment_model", "distilbert")

## Benchmarking every configuration

One fixed measurement protocol for all of them, because latency numbers taken under
different thread counts or batch sizes aren't comparable. CPU only, two threads pinned
(roughly what the free hosting tier gives you), 128 tokens, accuracy over the full test
set in batches of 32, latency one request at a time.

This also adds the fp32-ONNX rows, which separate the runtime change from the
quantization change. Without them you can't tell which one bought the speedup.

In [ ]:
# Shared standalone helpers
def _load_split(seed=42, test_size=0.2):
    """Rebuild the EXACT train/test split used everywhere else in this notebook."""
    import kagglehub, pandas as pd
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import train_test_split
    p = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
    df = pd.read_csv(p + '/Combined Data.csv', index_col=0)
    df.dropna(inplace=True)
    le = LabelEncoder()
    df['labels'] = le.fit_transform(df['status'])
    tr, te = train_test_split(df, test_size=test_size, stratify=df['labels'],
                              random_state=seed)
    return tr, te, le

def _deployed_session(model_dir="./distilbert_onnx_int8", threads=2):
    """Load the deployed quantized model exactly as the Space serves it."""
    import os, json as _json
    import onnxruntime as ort
    from transformers import AutoTokenizer
    graphs = [f for f in os.listdir(model_dir) if f.endswith(".onnx")]
    name = "model_quantized.onnx" if "model_quantized.onnx" in graphs else graphs[0]
    cfg = _json.load(open(os.path.join(model_dir, "config.json")))
    labels = [cfg["id2label"][str(i)] for i in range(len(cfg["id2label"]))]
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    sess = ort.InferenceSession(os.path.join(model_dir, name), so,
                                providers=["CPUExecutionProvider"])
    return sess, AutoTokenizer.from_pretrained(model_dir), labels, \
           {i.name for i in sess.get_inputs()}

def _softmax(z):
    import numpy as np
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

# CONFUSION MATRIX, DEPLOYED DistilBERT int8
# Standalone. Requires ./distilbert_onnx_int8
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score)
from tqdm.auto import tqdm

MODEL_DIR, MAX_LEN, BATCH, THREADS = "./distilbert_onnx_int8", 128, 32, 2

_, test_df, le = _load_split()
CLASSES = list(le.classes_)
sess, tok, model_labels, IN = _deployed_session(MODEL_DIR, THREADS)
if model_labels != CLASSES:
    print("WARNING: label order mismatch between model config and data encoder!")
    print("  model:", model_labels, "\n  data :", CLASSES)

texts = test_df["statement"].astype(str).tolist()
y_true = test_df["labels"].values
preds = []
for i in tqdm(range(0, len(texts), BATCH), desc="Classifying"):
    enc = tok(texts[i:i + BATCH], return_tensors="np", truncation=True,
              padding=True, max_length=MAX_LEN)
    feed = {k: v.astype(np.int64) for k, v in enc.items() if k in IN}
    preds.append(sess.run(None, feed)[0].argmax(-1))
y_pred = np.concatenate(preds)

print(f"\naccuracy {accuracy_score(y_true, y_pred):.4f}   "
      f"macro-F1 {f1_score(y_true, y_pred, average='macro'):.4f}   "
      f"n={len(y_true):,}  (batch={BATCH})\n")
print(classification_report(y_true, y_pred, target_names=CLASSES,
                            digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
cmn = cm / cm.sum(axis=1, keepdims=True)

# Modified Plotting Block
fig, ax = plt.subplots(1, 2, figsize=(21, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0])
ax[0].set(xlabel='Predicted', ylabel='Actual',
          title='Confusion Matrix - DistilBERT int8 Sentiment Analysis')

sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Purples', vmin=0, vmax=1,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[1])
ax[1].set(xlabel='Predicted', ylabel='Actual',
          title='Row-normalized (per-class recall on diagonal)')
plt.tight_layout()

plt.savefig("fig_confusion_deployed.pdf", bbox_inches="tight")
plt.savefig("fig_confusion_deployed.png", dpi=220, bbox_inches="tight")
print("Saved fig_confusion_deployed.pdf / .png")
plt.show()

s = CLASSES.index("Suicidal")
print("\nWhere Suicidal items go when the model gets them wrong:")
row = pd.DataFrame({"count": cm[s], "share of Suicidal": cmn[s].round(4)},
                   index=CLASSES).sort_values("count", ascending=False)
print(row.to_string())
print(f"\nSuicidal -> Normal (no flag raised at all): {cm[s, CLASSES.index('Normal')]} "
      f"of {cm[s].sum()} = {100*cmn[s, CLASSES.index('Normal')]:.2f}%")
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv("phase2_B_confusion_counts.csv")

### Model size

Careful here: summing a whole checkpoint directory counts the optimizer state, which
is roughly 2x the weights and never gets deployed. Comparing an inflated BERT directory
against a weights-only DistilBERT one exaggerates the compression ratio a lot. This
separates deployable bytes from training artifacts and recomputes the ratio on one basis.

In [ ]:
# 1.2 MODEL SIZE AUDIT
# Standalone. Only reads directories; loads no data and runs no inference.
import os, json
import pandas as pd

DIRS = ["./bert_sentiment_model", "./distilbert_sentiment_model",
        "./bert_onnx", "./bert_onnx_int8",
        "./distilbert_onnx", "./distilbert_onnx_int8",
        "./results", "./distil_results"]        # training output dirs, for context

WEIGHT_EXT   = {".safetensors", ".bin", ".onnx", ".data", ".h5", ".msgpack"}
TRAINING_ONLY = {"optimizer.pt", "optimizer.bin", "scheduler.pt", "rng_state.pth",
                 "trainer_state.json", "training_args.bin", "scaler.pt"}

def _mb(n): return n / 1e6

def audit(path):
    if not os.path.isdir(path):
        return None
    files, deploy, training, ckpt = [], 0, 0, 0
    for root, dirs, fnames in os.walk(path):
        in_ckpt = os.path.basename(root).startswith("checkpoint")
        for f in fnames:
            fp = os.path.join(root, f)
            size = os.path.getsize(fp)
            rel = os.path.relpath(fp, path)
            if in_ckpt:
                ckpt += size; bucket = "checkpoint"
            elif f in TRAINING_ONLY:
                training += size; bucket = "training-only"
            else:
                deploy += size; bucket = "deployable"
            files.append({"file": rel, "MB": round(_mb(size), 2), "bucket": bucket})
    files.sort(key=lambda r: -r["MB"])

    n_params, expected_fp32 = None, None
    cfg_path = os.path.join(path, "config.json")
    if os.path.exists(cfg_path):
        cfg = json.load(open(cfg_path))
        h  = cfg.get("hidden_size") or cfg.get("dim")
        L  = cfg.get("num_hidden_layers") or cfg.get("n_layers")
        V  = cfg.get("vocab_size")
        ff = cfg.get("intermediate_size") or cfg.get("hidden_dim")
        P  = cfg.get("max_position_embeddings")
        if all(x is not None for x in (h, L, V, ff, P)):
            emb = V * h + P * h + 2 * h
            per_layer = 4 * h * h + 4 * h + 2 * h * ff + h + ff + 4 * h
            n_params = emb + L * per_layer
            expected_fp32 = _mb(n_params * 4)

    return {"path": path,
            "total_MB": round(_mb(deploy + training + ckpt), 1),
            "deployable_MB": round(_mb(deploy), 1),
            "training_only_MB": round(_mb(training), 1),
            "checkpoint_MB": round(_mb(ckpt), 1),
            "approx_params_M": round(n_params / 1e6, 1) if n_params else None,
            "expected_fp32_MB": round(expected_fp32, 1) if expected_fp32 else None,
            "files": files}

results = [r for r in (audit(d) for d in DIRS) if r]

tbl = pd.DataFrame([{k: v for k, v in r.items() if k != "files"} for r in results])
print("=" * 100)
print("DIRECTORY AUDIT")
print("=" * 100)
print(tbl.to_string(index=False))

print("\n" + "=" * 100)
print("FILE BREAKDOWN (files >= 1 MB)")
print("=" * 100)
for r in results:
    print(f"\n{r['path']}   total {r['total_MB']} MB")
    for f in r["files"]:
        if f["MB"] >= 1.0:
            print(f"    {f['MB']:9.2f} MB  [{f['bucket']:>13}]  {f['file']}")
    hidden = sum(1 for f in r["files"] if f["MB"] < 1.0)
    if hidden:
        print(f"    ... {hidden} files under 1 MB")

print("\n" + "=" * 100)
print("SANITY CHECK: deployable size vs parameter count")
print("=" * 100)
for r in results:
    if r["expected_fp32_MB"]:
        ratio = r["deployable_MB"] / r["expected_fp32_MB"]
        flag = ""
        if ratio > 1.5:
            flag = "  <-- LARGER than fp32 weights: extra artifacts still counted"
        elif ratio < 0.6:
            flag = "  <-- smaller than fp32: quantized or weight file missing"
        print(f"  {r['path']:32} {r['approx_params_M']:>6.1f}M params  "
              f"expected fp32 {r['expected_fp32_MB']:>7.1f} MB  "
              f"deployable {r['deployable_MB']:>7.1f} MB  "
              f"ratio {ratio:.2f}{flag}")

print("\n" + "=" * 100)
print("COMPRESSION RATIOS: recomputed on a single consistent basis")
print("=" * 100)
by = {r["path"]: r for r in results}
ref = by.get("./bert_sentiment_model")
tgt = by.get("./distilbert_onnx_int8")
if ref and tgt:
    for basis in ("total_MB", "deployable_MB"):
        print(f"  basis = {basis:16} BERT {ref[basis]:>8.1f} MB  ->  "
              f"DistilBERT int8 {tgt[basis]:>7.1f} MB  =  "
              f"{ref[basis]/tgt[basis]:.1f}x smaller")
    print("\n  Report the `deployable_MB` ratio. `total_MB` counts optimizer state")
    print("  and intermediate checkpoints, which are never shipped to the Space.")
else:
    print("  (need ./bert_sentiment_model and ./distilbert_onnx_int8 present)")

pd.DataFrame([{k: v for k, v in r.items() if k != "files"}
              for r in results]).to_csv("phase1_2_size_audit.csv", index=False)
print("\nWrote phase1_2_size_audit.csv")

## Frequency baselines

What does the transformer's cost actually buy? Bag-of-Words and TF-IDF under the same
protocol, scored on both the full and leakage-filtered test sets.

In [ ]:
# FREQUENCY BASELINES: full + clean subsets, predictions saved
# Standalone. Needs phase2_D_leakage_mask.npz. No model folders required.
import os, time, pickle, warnings, numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
warnings.filterwarnings("ignore")

THREADS, N_LAT, SEED = 2, 300, 42
MAX_FEATS, NGRAM = 50_000, (1, 2)
os.environ["OMP_NUM_THREADS"] = str(THREADS)
W_MISS_HIGH, W_UNDER_HIGH, W_MISS_ELEVATED, W_CONFUSE, W_FALSE_ALARM = 10., 3., 3., 1., 1.

import kagglehub
p = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
df = pd.read_csv(p + '/Combined Data.csv', index_col=0).dropna()
le = LabelEncoder(); df['labels'] = le.fit_transform(df['status'])
CLASSES = list(le.classes_)
NORMAL_ID, SUICIDAL_ID = CLASSES.index("Normal"), CLASSES.index("Suicidal")
tr, te = train_test_split(df, test_size=0.2, stratify=df['labels'], random_state=SEED)
Xtr, ytr = tr['statement'].astype(str).tolist(), tr['labels'].values
Xte, yte = te['statement'].astype(str).tolist(), te['labels'].values
clean = ~np.load("phase2_D_leakage_mask.npz")["strict"]
print(f"train {len(ytr):,} | test {len(yte):,} | clean {int(clean.sum()):,}")

def triage_cost(y_true, y_pred):
    C = np.zeros((7, 7))
    for i in range(7):
        for j in range(7):
            if i == j: continue
            if i == SUICIDAL_ID:   C[i, j] = W_MISS_HIGH if j == NORMAL_ID else W_UNDER_HIGH
            elif i == NORMAL_ID:   C[i, j] = W_FALSE_ALARM
            else:                  C[i, j] = W_MISS_ELEVATED if j == NORMAL_ID else W_CONFUSE
    cm = confusion_matrix(y_true, y_pred, labels=range(7))
    hard = int(cm[SUICIDAL_ID, NORMAL_ID])
    return float((cm * C).sum()) / len(y_true), hard

CONFIGS = [
    ("BoW + LogisticRegression",
     CountVectorizer(max_features=MAX_FEATS, ngram_range=NGRAM),
     LogisticRegression(max_iter=1000, n_jobs=THREADS)),
    ("TF-IDF + LogisticRegression",
     TfidfVectorizer(max_features=MAX_FEATS, ngram_range=NGRAM, sublinear_tf=True),
     LogisticRegression(max_iter=1000, n_jobs=THREADS)),
    ("TF-IDF + MultinomialNB",
     TfidfVectorizer(max_features=MAX_FEATS, ngram_range=NGRAM, sublinear_tf=True),
     MultinomialNB()),
]

rows, perclass, preds = [], {}, {}
for name, vec, clf in CONFIGS:
    print(f"--- {name} ---")
    pipe = Pipeline([("vec", vec), ("clf", clf)]).fit(Xtr, ytr)
    y_pred = pipe.predict(Xte); preds[name.replace(" ", "_")] = y_pred

    for t in Xte[:10]: pipe.predict([t])
    t0 = time.perf_counter()
    for t in Xte[:N_LAT]: pipe.predict([t])
    ms = (time.perf_counter() - t0) / N_LAT * 1000
    t0 = time.perf_counter(); pipe.predict(Xte[:N_LAT])
    thru = N_LAT / (time.perf_counter() - t0)
    mb = len(pickle.dumps(pipe)) / 1e6

    row = {"Model": name, "ms/req": round(ms, 2), "req/s": round(thru, 1),
           "MB": round(mb, 1)}
    for sub, msk in [("full", np.ones_like(clean)), ("clean", clean)]:
        cost, hard = triage_cost(yte[msk], y_pred[msk])
        row[f"Acc ({sub})"]     = round(accuracy_score(yte[msk], y_pred[msk]), 4)
        row[f"MacroF1 ({sub})"] = round(f1_score(yte[msk], y_pred[msk],
                                        average="macro", zero_division=0), 4)
        row[f"SuicR ({sub})"]   = round(recall_score(yte[msk], y_pred[msk],
                                        average=None, zero_division=0)[SUICIDAL_ID], 4)
        row[f"ETC ({sub})"]     = round(cost, 4)
        row[f"HardMiss ({sub})"] = hard
    rows.append(row)

    rep = classification_report(yte[clean], y_pred[clean], labels=range(7),
                                target_names=CLASSES, output_dict=True, zero_division=0)
    perclass[name] = {c: round(rep[c]["f1-score"], 4) for c in CLASSES}
    print(f"    clean acc={row['Acc (clean)']}  macroF1={row['MacroF1 (clean)']}  "
          f"{row['ms/req']} ms/req  {row['MB']} MB")

summary = pd.DataFrame(rows)
print("\n" + "=" * 96)
print(f"FREQUENCY BASELINES: CPU, {THREADS} threads")
print("=" * 96)
print(summary.to_string(index=False))
print("\nPer-class F1 (clean subset):")
print(pd.DataFrame(perclass).reindex(CLASSES).to_string())

summary.to_csv("baselines_full_and_clean.csv", index=False)
pd.DataFrame(perclass).reindex(CLASSES).to_csv("baselines_perclass_clean.csv")
np.savez("baselines_preds.npz", labels=yte, **preds)
print("\nWrote baselines_full_and_clean.csv, baselines_perclass_clean.csv, "
      "baselines_preds.npz")

## Calibration

The app shows probabilities to users, so they should mean something. Expected
calibration error, Brier score, and negative log-likelihood for each variant.

One thing to watch: ECE only measures whether stated confidence matches observed
accuracy, so a model can score well on it by being uniformly unsure. Brier and NLL
reward being right as well as being calibrated, which is why all three are here.

In [ ]:
# Shared standalone helpers (duplicated in each Phase-1 cell on purpose)
def _load_split(seed=42, test_size=0.2):
    """Rebuild the EXACT train/test split used everywhere else in this notebook."""
    import kagglehub, pandas as pd
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import train_test_split
    p = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
    df = pd.read_csv(p + '/Combined Data.csv', index_col=0)
    df.dropna(inplace=True)
    le = LabelEncoder()
    df['labels'] = le.fit_transform(df['status'])
    tr, te = train_test_split(df, test_size=test_size, stratify=df['labels'],
                              random_state=seed)
    return tr, te, le

# 1.7 CALIBRATION
# Standalone. Requires the model folders (ONNX variants exported if missing).
import os, gc, warnings
import numpy as np, pandas as pd, torch
import onnxruntime as ort
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

MAX_LEN, THREADS, BATCH, SEED = 128, 2, 32, 42
N_BINS = 15
N_EVAL = None       # None = full test set
torch.set_num_threads(THREADS)

train_df, test_df, le = _load_split(seed=SEED)
CLASSES = list(le.classes_)
texts  = test_df["statement"].astype(str).tolist()
labels = test_df["labels"].values
if N_EVAL:
    texts, labels = texts[:N_EVAL], labels[:N_EVAL]
print(f"Test set: {len(texts):,}\n")


def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def ece_mce(conf, correct, n_bins=N_BINS):
    """Equal-width binning on top-1 confidence."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, mce, rows = 0.0, 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            rows.append({"lo": lo, "hi": hi, "n": 0, "conf": np.nan, "acc": np.nan})
            continue
        c, a = conf[m].mean(), correct[m].mean()
        gap = abs(c - a)
        ece += (m.sum() / len(conf)) * gap
        mce = max(mce, gap)
        rows.append({"lo": lo, "hi": hi, "n": int(m.sum()),
                     "conf": float(c), "acc": float(a)})
    return ece, mce, pd.DataFrame(rows)


def brier_multiclass(probs, y):
    onehot = np.zeros_like(probs)
    onehot[np.arange(len(y)), y] = 1.0
    return float(((probs - onehot) ** 2).sum(axis=1).mean())


def _pick(path, preferred):
    files = [f for f in os.listdir(path) if f.endswith(".onnx")]
    return preferred if preferred in files else files[0]


def get_probs(kind, path, onnx_name):
    tok = AutoTokenizer.from_pretrained(path)
    out = []
    if kind == "pt":
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cpu").eval()
        for i in tqdm(range(0, len(texts), BATCH), leave=False):
            enc = tok(texts[i:i + BATCH], return_tensors="pt", truncation=True,
                      padding=True, max_length=MAX_LEN)
            with torch.no_grad():
                out.append(m(**enc).logits.numpy())
        del m
    else:
        so = ort.SessionOptions(); so.intra_op_num_threads = THREADS
        sess = ort.InferenceSession(os.path.join(path, _pick(path, onnx_name)), so,
                                    providers=["CPUExecutionProvider"])
        IN = {i.name for i in sess.get_inputs()}
        for i in tqdm(range(0, len(texts), BATCH), leave=False):
            enc = tok(texts[i:i + BATCH], return_tensors="np", truncation=True,
                      padding=True, max_length=MAX_LEN)
            feed = {k: v.astype(np.int64) for k, v in enc.items() if k in IN}
            out.append(sess.run(None, feed)[0])
        del sess
    gc.collect()
    return softmax(np.concatenate(out).astype(np.float64))


VARIANTS = [
    ("BERT fp32 (PyTorch)",       "pt",   "./bert_sentiment_model",       None),
    ("BERT int8 (ONNX)",          "onnx", "./bert_onnx_int8",             "model_quantized.onnx"),
    ("DistilBERT fp32 (PyTorch)", "pt",   "./distilbert_sentiment_model", None),
    ("DistilBERT int8 (ONNX)",    "onnx", "./distilbert_onnx_int8",       "model_quantized.onnx"),
]
VARIANTS = [v for v in VARIANTS if os.path.isdir(v[2])]

rows, bins_store, probs_store = [], {}, {}
for name, kind, path, onnx_name in VARIANTS:
    print(f"--- {name} ---")
    probs = get_probs(kind, path, onnx_name)
    probs_store[name] = probs
    pred = probs.argmax(1)
    conf = probs.max(1)
    correct = (pred == labels).astype(float)

    ece, mce, bdf = ece_mce(conf, correct)
    bins_store[name] = bdf
    rows.append({
        "Model": name,
        "Acc": round(accuracy_score(labels, pred), 4),
        "MeanConf": round(float(conf.mean()), 4),
        "ECE": round(ece, 4),
        "MCE": round(mce, 4),
        "Brier": round(brier_multiclass(probs, labels), 4),
        "NLL": round(float(-np.log(np.clip(
            probs[np.arange(len(labels)), labels], 1e-12, None)).mean()), 4),
        "conf>=0.99 %": round(100 * float((conf >= 0.99).mean()), 2),
        "conf>=0.999 %": round(100 * float((conf >= 0.999).mean()), 2),
        "max conf": round(float(conf.max()), 6),
    })
    print(f"    ECE={ece:.4f}  MCE={mce:.4f}  mean conf={conf.mean():.4f}  "
          f"acc={rows[-1]['Acc']}  overconfidence={conf.mean()-correct.mean():+.4f}")

summary = pd.DataFrame(rows)
print("\n" + "=" * 88)
print(f"CALIBRATION: CPU, MAX_LEN={MAX_LEN}, n={len(texts):,}, {N_BINS} bins")
print("=" * 88)
print(summary.to_string(index=False))
print("\nMeanConf - Acc > 0 means the model is overconfident on average.")

# Reliability diagrams
n = len(VARIANTS)
fig, axes = plt.subplots(2, n, figsize=(4.2 * n, 7.6), squeeze=False)
for k, (name, *_ ) in enumerate(VARIANTS):
    bdf = bins_store[name].dropna()
    
    # NEW FILTER: Only graph bins where n > 20
    bdf = bdf[bdf["n"] > 20] 
    
    centers = (bdf["lo"] + bdf["hi"]) / 2
    ax = axes[0][k]
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect")
    ax.bar(centers, bdf["acc"], width=1.0 / N_BINS * 0.9, edgecolor="black",
           alpha=0.75, label="accuracy")
    ax.plot(centers, bdf["conf"], "o-", color="crimson", ms=3, label="confidence")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f"{name}\nECE={summary.iloc[k]['ECE']:.4f}", fontsize=10)
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy" if k == 0 else "")
    if k == 0:
        ax.legend(fontsize=8)
    ax2 = axes[1][k]
    ax2.hist(probs_store[name].max(1), bins=N_BINS, range=(0, 1),
             edgecolor="black", alpha=0.8)
    ax2.set_yscale("log")
    ax2.set_xlabel("top-1 confidence")
    ax2.set_ylabel("count (log)" if k == 0 else "")
    ax2.set_title("confidence distribution", fontsize=9)
plt.tight_layout()
plt.savefig("phase1_7_reliability.png", dpi=200, bbox_inches="tight")
print("\nSaved phase1_7_reliability.png")
plt.show()

summary.to_csv("phase1_7_calibration.csv", index=False)
for name, bdf in bins_store.items():
    # NEW FILTER: Ensure the saved CSVs also exclude the noisy/small bins
    bdf_filtered = bdf.dropna()
    bdf_filtered = bdf_filtered[bdf_filtered["n"] > 20]
    bdf_filtered.to_csv(f"phase1_7_bins_{name.split(' ')[0]}_"
               f"{'int8' if 'int8' in name else 'fp32'}.csv", index=False)

np.savez_compressed("phase1_7_probs.npz", labels=labels,
                    **{k.replace(" ", "_"): v for k, v in probs_store.items()})
print("Wrote phase1_7_calibration.csv, per-bin CSVs, phase1_7_probs.npz")

# Per-class calibration for the deployed model
dep = "DistilBERT int8 (ONNX)"
if dep in probs_store:
    print("\n" + "=" * 88)
    print(f"PER-CLASS CALIBRATION: {dep}")
    print("=" * 88)
    p = probs_store[dep]; pred = p.argmax(1); conf = p.max(1)
    out = []
    for i, c in enumerate(CLASSES):
        m = pred == i
        if m.sum() == 0:
            continue
        e, _, _ = ece_mce(conf[m], (labels[m] == i).astype(float))
        out.append({"predicted class": c, "n": int(m.sum()),
                    "mean conf": round(float(conf[m].mean()), 4),
                    "precision": round(float((labels[m] == i).mean()), 4),
                    "ECE": round(e, 4)})
    print(pd.DataFrame(out).to_string(index=False))
    print("\nA class whose mean confidence far exceeds its precision is one where")
    print("the displayed percentage most misleads a lay user.")

Reliability diagrams. Bins holding only a handful of samples produce wild accuracy
values and dominate the maximum calibration error, so low-support bins are dropped
and the per-bin counts are annotated.

In [ ]:
# RELIABILITY DIAGRAM (corrected: minimum bin support)
# Standalone. Requires only phase1_7_probs.npz from cell 1.7.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

N_BINS, MIN_BIN_N = 15, 20
z = np.load("phase1_7_probs.npz")
labels = z["labels"]
MODELS = [k for k in z.files if k != "labels"]
ORDER = ["BERT_fp32_(PyTorch)", "BERT_int8_(ONNX)",
         "DistilBERT_fp32_(PyTorch)", "DistilBERT_int8_(ONNX)"]
MODELS = [m for m in ORDER if m in MODELS] + [m for m in MODELS if m not in ORDER]

def bin_stats(conf, correct, n_bins=N_BINS):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        rows.append({"lo": lo, "hi": hi, "n": int(m.sum()),
                     "conf": float(conf[m].mean()) if m.sum() else np.nan,
                     "acc":  float(correct[m].mean()) if m.sum() else np.nan})
    return pd.DataFrame(rows)

def ece_mce(df, total, min_n=MIN_BIN_N):
    ok = df[df["n"] > 0]
    ece = float(((ok["n"] / total) * (ok["conf"] - ok["acc"]).abs()).sum())
    sup = df[df["n"] >= min_n]
    mce_all = float((ok["conf"] - ok["acc"]).abs().max())
    mce_sup = float((sup["conf"] - sup["acc"]).abs().max()) if len(sup) else np.nan
    return ece, mce_all, mce_sup

rows, store = [], {}
for m in MODELS:
    p = z[m]; pred = p.argmax(1); conf = p.max(1)
    correct = (pred == labels).astype(float)
    df = bin_stats(conf, correct); store[m] = df
    ece, mce_all, mce_sup = ece_mce(df, len(conf))
    dropped = int(df[(df["n"] > 0) & (df["n"] < MIN_BIN_N)]["n"].sum())
    rows.append({"Model": m.replace("_", " "),
                 "Acc": round(float(correct.mean()), 4),
                 "MeanConf": round(float(conf.mean()), 4),
                 "ECE": round(ece, 4),
                 "MCE (all bins)": round(mce_all, 4),
                 f"MCE (n>={MIN_BIN_N})": round(mce_sup, 4),
                 "items in dropped bins": dropped,
                 "% of test set": round(100 * dropped / len(conf), 3)})

summary = pd.DataFrame(rows)
print("=" * 100)
print(f"CALIBRATION, low-support bins excluded from MCE (min n = {MIN_BIN_N})")
print("=" * 100)
print(summary.to_string(index=False))
print("\nReport the supported MCE. The all-bins column is dominated by bins")
print("holding a handful of items and is not a stable statistic.")
summary.to_csv("phase2_A_calibration_corrected.csv", index=False)

n = len(MODELS)
fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.5), squeeze=False)
for k, m in enumerate(MODELS):
    df = store[m]
    plot_df = df[df["n"] >= MIN_BIN_N]
    centers = (plot_df["lo"] + plot_df["hi"]) / 2
    ax = axes[0][k]
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="perfect", zorder=1)
    ax.bar(centers, plot_df["acc"], width=0.9 / N_BINS, edgecolor="black",
           lw=0.5, alpha=0.8, label="accuracy", zorder=2)
    ax.plot(centers, plot_df["conf"], "o-", color="crimson", ms=3, lw=1.2,
            label="confidence", zorder=3)
    for c, a, nn in zip(centers, plot_df["acc"], plot_df["n"]):
        ax.text(c, a + 0.02, f"{int(nn)}", ha="center", va="bottom",
                fontsize=5.5, rotation=90, color="#34495E")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.06)
    r = summary.iloc[k]
    ax.set_title(f"{m.replace('_',' ')}\nECE={r['ECE']:.4f}  "
                 f"MCE={r[f'MCE (n>={MIN_BIN_N})']:.4f}", fontsize=9)
    ax.set_xlabel("confidence")
    ax.set_ylabel("accuracy" if k == 0 else "")
    if k == 0:
        ax.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.savefig("fig_reliability.pdf", bbox_inches="tight")
plt.savefig("fig_reliability.png", dpi=220, bbox_inches="tight")
print("\nSaved fig_reliability.pdf / .png  (bar annotations = items per bin)")
plt.show()

## Confusion matrix for the deployed model

Row-normalized, so the diagonal reads directly as per-class recall. With 3,269 Normal
items against 215 Personality disorder ones, a raw-count matrix makes the minority
classes invisible.

In [ ]:
# Standalone: confusion matrix for the deployed DistilBERT int8 model
# NOTE: no pip install here on purpose. Everything needed was installed by the
# one-time setup cell at the top. A bare `pip install transformers` would pull
# transformers 5.x and break the pinned optimum/ONNX stack.

import json, os, numpy as np, pandas as pd
import onnxruntime as ort
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, f1_score)
from tqdm.auto import tqdm

MODEL_DIR = "./distilbert_onnx_int8"   # <-- edit if your folder lives elsewhere
MAX_LEN, BATCH = 128, 64               # MAX_LEN must match fine-tuning/deployment

# 1. Load the quantized model + its label mapping
onnx_files = [f for f in os.listdir(MODEL_DIR) if f.endswith(".onnx")]
if not onnx_files:
    raise FileNotFoundError(f"No .onnx file in {MODEL_DIR}")
onnx_name = "model_quantized.onnx" if "model_quantized.onnx" in onnx_files else onnx_files[0]
print(f"Using graph: {onnx_name}")

with open(os.path.join(MODEL_DIR, "config.json")) as f:
    cfg = json.load(f)
model_labels = [cfg["id2label"][str(i)] for i in range(len(cfg["id2label"]))]

# 2. Rebuild the EXACT original test split
import kagglehub
p = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
df = pd.read_csv(p + '/Combined Data.csv', index_col=0)
df.dropna(inplace=True)
le = LabelEncoder()
df['labels'] = le.fit_transform(df['status'])
_, test_df = train_test_split(df, test_size=0.2, stratify=df['labels'], random_state=42)
CLASSES = list(le.classes_)
print(f"Test set: {len(test_df)} samples")

if model_labels != CLASSES:
    print("WARNING: label order mismatch, axis names may be wrong!")
    print("  model:", model_labels)
    print("  data :", CLASSES)

# 3. Run inference
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
so = ort.SessionOptions(); so.intra_op_num_threads = max(1, os.cpu_count() or 1)
sess = ort.InferenceSession(os.path.join(MODEL_DIR, onnx_name), so,
                            providers=["CPUExecutionProvider"])
IN = {i.name for i in sess.get_inputs()}   # DistilBERT: input_ids + attention_mask only

texts, y_true = test_df["statement"].astype(str).tolist(), test_df["labels"].values
preds = []
for i in tqdm(range(0, len(texts), BATCH), desc="Classifying"):
    enc = tok(texts[i:i+BATCH], return_tensors="np", truncation=True,
              padding=True, max_length=MAX_LEN)
    feed = {k: v.astype(np.int64) for k, v in enc.items() if k in IN}
    preds.append(sess.run(None, feed)[0].argmax(-1))
y_pred = np.concatenate(preds)

# 4. Metrics + confusion matrix
print(f"\nAccuracy {accuracy_score(y_true, y_pred):.4f} | "
      f"Macro-F1 {f1_score(y_true, y_pred, average='macro'):.4f}\n")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(1, 2, figsize=(21, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0])
ax[0].set(xlabel='Predicted', ylabel='Actual',
          title='Confusion Matrix - DistilBERT int8 Sentiment Analysis')
cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Purples', vmin=0, vmax=1,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[1])
ax[1].set(xlabel='Predicted', ylabel='Actual',
          title='Row-normalized (per-class recall on diagonal)')
plt.tight_layout()
plt.savefig("confusion_matrix_distilbert_int8.png", dpi=300, bbox_inches='tight')
plt.show()

## Edge-case suite

Aggregate accuracy doesn't say much about how the model handles figurative language.
Fifty probes across five phenomena (hyperbole, idiom, sarcasm, temporal blindness,
and minimized distress) each with the label a human reader would assign.

The failures are lopsided in a way the headline metrics don't show at all.

In [ ]:
import os
import json as _json
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer

# 1. ARRAYS FOR TEST CASES, EXPECTED RESULTS, AND CATEGORY TYPES
TEST_TEXTS = [
    # Hyperbole
    "I'm so hungry I could eat a horse right now.",
    "This deadline is actually going to kill me, I've got way too much on my plate this week.",
    "I laughed so hard I nearly died, that video was hilarious.",
    "My heart is pounding out of my chest, I'm so nervous about this exam I could throw up.",
    "I feel like the weight of the entire world is crushing me, I can't shake this sadness no matter what I do.",
    "I have a thousand things to do and only twenty-four hours in the day, I'm losing it.",
    "That concert was insane, best night of my entire life, I'm still buzzing.",
    "I'm so anxious about the flight tomorrow I feel like I'm going to explode.",
    "I feel like I'm carrying around a two hundred pound weight of sadness that never goes away no matter how much I try to shake it off.",
    "I have so much on my plate right now I feel like I'm about to combust, this week has been absolute chaos.",

    # Idiom
    "I killed two birds with one stone today, finished my report and picked up groceries.",
    "That joke had me dying, so funny.",
    "I'm swamped at work this week, drowning in emails and back-to-back meetings.",
    "My stomach's in knots waiting for the test results to come back.",
    "I've been feeling down in the dumps for weeks now, nothing seems to lift my mood.",
    "Break a leg tonight, you're going to do great.",
    "I'm running on fumes trying to juggle work and the kids this week.",
    "I've been walking on eggshells all week worried about the layoffs at work.",
    "One week I'm on top of the world starting five new projects and the next I can't even get off the couch, it's like living with two different people in my head.",
    "I burn bridges the second I feel someone pulling away, better to torch it myself than wait around to get left, that's just how I'm wired I guess.",

    # Sarcasm
    "Oh great, another all-nighter, exactly what I needed with three deadlines this week.",
    "Wonderful, it's raining again the one day I decided to walk to work.",
    "Fantastic, my hands are shaking again right before the presentation, love that for me.",
    "Great, another day where I can't find the motivation to get out of bed, really thriving over here.",
    "Perfect, spilled coffee all over my shirt right before the meeting, off to a great start.",
    "Love how my inbox somehow doubled overnight, definitely going to be a relaxing week.",
    "Oh good, my mind is racing through worst case scenarios again at 2am, exactly what I needed.",
    "Sure, I'm 'doing great,' that's why I've cancelled plans with everyone three weekends in a row.",
    "Oh I'm SO fine, definitely not lying awake calculating how much easier it would be for everyone if I just wasn't here anymore, nothing to see here.",
    "Great, another 'friend' who disappears the second things get hard, must be something about me right? Awesome pattern I've got going, really love collecting people just to lose them.",

    # Temporal blindness
    "I used to struggle with anxiety attacks in college, but I haven't had one in years and feel completely fine now.",
    "I went through a rough depressive episode after my divorce, but that was five years ago and I'm honestly in a really good place now.",
    "I've been anxious about this move for weeks now and it's still eating at me every single day.",
    "Ever since the new project started last month, I've been stressed out non-stop and it hasn't let up.",
    "I've felt this heavy sadness since the breakup two months ago and it still hasn't gone away.",
    "I had panic attacks daily back in high school, but that stopped once I graduated and I've been panic-free for almost a decade.",
    "I had a manic episode last year and I'm still cycling between highs and crashes most months.",
    "I used to have really bad social anxiety, but after a few years of practice I actually enjoy parties now.",
    "I told my therapist I 'used to' think about not waking up, but honestly some nights it's still there, I just don't say it out loud anymore.",
    "I used to tell people my relationships 'just didn't work out,' but looking back, every single one ended the same way, me either clinging so hard they ran or pushing them away before they could leave first, and honestly that hasn't really changed.",

    # Minimized distress
    "I've been swamped with work lately, staying late most nights and barely keeping up, but I think I just need a lighter week.",
    "I'm okay I guess, just haven't felt like myself in weeks and everything feels kind of pointless.",
    "It's nothing, I just get really panicky before flights, my heart races and I can't sit still.",
    "No big deal, just been crying more than usual and struggling to get out of bed most mornings.",
    "It's fine honestly, just thinking too much right now and feeling constantly worried about stuff.",
    "Not a big thing, I just can't stop worrying about everything lately, my mind won't slow down.",
    "It's nothing serious, I just go through these weeks where I barely sleep and work way too much, then crash hard after.",
    "I'm managing, just feel numb most days and nothing really excites me like it used to.",
    "It's nothing really, I've just been having thoughts about wanting to end my life, but don't make a big deal out of it, I'll be fine.",
    "I don't mean to bother anyone, and I know everyone here has much bigger problems than me, but I've been feeling a bit overwhelmed lately. Sorry for wasting your time.",
]

EXPECTED_LABELS = [
    # Hyperbole
    "Normal",
    "Stress",
    "Normal",
    "Anxiety",
    "Depression",
    "Stress",
    "Normal",
    "Anxiety",
    "Depression",
    "Stress",

    # Idiom
    "Normal",
    "Normal",
    "Stress",
    "Anxiety",
    "Depression",
    "Normal",
    "Stress",
    "Anxiety",
    "Bipolar",
    "Personality disorder",

    # Sarcasm
    "Stress",
    "Normal",
    "Anxiety",
    "Depression",
    "Normal",
    "Stress",
    "Anxiety",
    "Depression",
    "Suicidal",
    "Personality disorder",

    # Temporal blindness
    "Normal",
    "Normal",
    "Anxiety",
    "Stress",
    "Depression",
    "Normal",
    "Bipolar",
    "Normal",
    "Suicidal",
    "Personality disorder",

    # Minimized distress
    "Stress",
    "Depression",
    "Anxiety",
    "Depression",
    "Anxiety",
    "Anxiety",
    "Stress",
    "Depression",
    "Suicidal",
    "Stress",
]


TEST_TYPES = [
    # Hyperbole
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",
    "Hyperbole",

    # Idiom
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",
    "Idiom",

    # Sarcasm
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",
    "Sarcasm",

    # Temporal blindness
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",
    "Temporal blindness",

    # Minimized distress
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
    "Minimized distress",
]

# 2. BARE BONES CLASSIFICATION & EVALUATION
MODEL_DIR = "./distilbert_onnx_int8"
MAX_LEN = 128

def _softmax(x):
    """Compute softmax values for a 1D numpy array."""
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def load_model():
    """Loads the quantized ONNX model and tokenizer."""
    graphs = [f for f in os.listdir(MODEL_DIR) if f.endswith(".onnx")]
    name = "model_quantized.onnx" if "model_quantized.onnx" in graphs else graphs[0]
    
    with open(os.path.join(MODEL_DIR, "config.json")) as f:
        cfg = _json.load(f)
    classes = [cfg["id2label"][str(i)] for i in range(len(cfg["id2label"]))]
    
    sess = ort.InferenceSession(os.path.join(MODEL_DIR, name), providers=["CPUExecutionProvider"])
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    valid_inputs = {i.name for i in sess.get_inputs()}
    
    return sess, tok, classes, valid_inputs

def classify(text, sess, tok, classes, valid_inputs):
    """Runs text through the model and returns (predicted label, confidence score)."""
    enc = tok([text], return_tensors="np", truncation=True, padding=True, max_length=MAX_LEN)
    feed = {k: v.astype(np.int64) for k, v in enc.items() if k in valid_inputs}
    
    logits = sess.run(None, feed)[0][0]
    probs = _softmax(logits)
    pred_idx = probs.argmax()
    
    return classes[pred_idx], float(probs[pred_idx])

# 3. RUN EVALUATION AND PRINT STATS
if len(TEST_TEXTS) > 0 and len(TEST_TEXTS) == len(EXPECTED_LABELS) == len(TEST_TYPES):
    print("Loading model...\n")
    sess, tok, CLASSES, IN = load_model()
    
    print("=" * 130)
    print("TEST CASE EVALUATION")
    print("=" * 130)
    
    total_correct = 0
    total_cases = len(TEST_TEXTS)
    type_stats = {}
    
    for text, actual, t_type in zip(TEST_TEXTS, EXPECTED_LABELS, TEST_TYPES):
        predicted, confidence = classify(text, sess, tok, CLASSES, IN)
        
        # Track Correctness
        is_correct = (predicted == actual)
        if is_correct:
            total_correct += 1
            
        # Track Per-Type Stats (including confidence sum for the mean)
        if t_type not in type_stats:
            type_stats[t_type] = {"correct": 0, "total": 0, "sum_conf": 0.0}
            
        type_stats[t_type]["total"] += 1
        type_stats[t_type]["sum_conf"] += confidence
        if is_correct:
            type_stats[t_type]["correct"] += 1
            
        # Formatting for display
        mark = "OK " if is_correct else "MISS"
        short_text = text if len(text) <= 50 else text[:47] + "..."
        
        print(f"[{mark}] type={t_type:20} actual={actual:12} pred={predicted:12} conf={confidence:.4f} text={short_text}")
    
    # Print Summary Statistics
    print("\n" + "=" * 70)
    print("SUMMARY STATISTICS")
    print("=" * 70)
    
    total_pct = (total_correct / total_cases) * 100
    print(f"Total Accuracy: {total_correct}/{total_cases} ({total_pct:.1f}%)\n")
    print(f"{'Category':<20} | {'Accuracy':<15} | {'Mean Confidence'}")
    print("-" * 70)
    
    for t_type, stats in type_stats.items():
        t_correct = stats["correct"]
        t_total = stats["total"]
        t_pct = (t_correct / t_total) * 100 if t_total > 0 else 0
        t_mean_conf = stats["sum_conf"] / t_total if t_total > 0 else 0
        
        acc_str = f"{t_correct}/{t_total} ({t_pct:.1f}%)"
        print(f"{t_type.title():<20} | {acc_str:<15} | {t_mean_conf:.4f}")
        
elif not (len(TEST_TEXTS) == len(EXPECTED_LABELS) == len(TEST_TYPES)) and len(TEST_TEXTS) > 0:
    print("Error: TEST_TEXTS, EXPECTED_LABELS, and TEST_TYPES must have the exact same number of items.")
else:
    print("Arrays are empty. Add items to TEST_TEXTS, EXPECTED_LABELS, and TEST_TYPES to run the evaluation.")

Full probability distribution for a single input, rather than just the argmax:

In [ ]:
def classify_full(text, sess, tok, classes, valid_inputs):
    """Returns the full probability distribution, not just argmax."""
    enc = tok([text], return_tensors="np", truncation=True, padding=True, max_length=MAX_LEN)
    feed = {k: v.astype(np.int64) for k, v in enc.items() if k in valid_inputs}
    logits = sess.run(None, feed)[0][0]
    probs = _softmax(logits)
    ranked = sorted(zip(classes, probs), key=lambda x: -x[1])
    return ranked

text = "I have a thousand things to do and only twenty-four hours in the day, I'm losing it."
for label, p in classify_full(text, sess, tok, CLASSES, IN):
    print(f"  {label:22} {p:.4f}")

## How much of this is noise?

Everything above comes from one training run on one split, so before claiming that any
two configurations differ, it's worth checking whether the gaps survive resampling.

Part 1 is a paired bootstrap over the test set: resample the test items, recompute the
metrics, and read off confidence intervals on the *differences*. Seconds to run, and it
answers "is this bigger than measurement noise?"

Part 2 retrains under different seeds, which is the more expensive question of whether
another training run would land in the same place. It's off by default.

In [ ]:
# VARIANCE AND UNCERTAINTY ESTIMATES
# Standalone. needs only phase1_1_preds.npz + phase2_D_leakage_mask.npz.
import os, gc, warnings, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score
warnings.filterwarnings("ignore")

PREDS   = "phase1_1_preds.npz"
MASK    = "phase2_D_leakage_mask.npz"
B       = 5000                    # bootstrap resamples
SEED    = 42
RUN_SEED_REPLICATION = True      # <-- set True for Part 2
SEEDS   = [42, 1337, 2024]
REF     = "BERT_fp32_(PyTorch)"
DEP     = "DistilBERT_int8_(ONNX)"
MLPERF  = 0.99

CLASSES = ['Anxiety','Bipolar','Depression','Normal','Personality disorder','Stress','Suicidal']
S = CLASSES.index("Suicidal")

def metrics(yt, yp):
    return dict(acc=accuracy_score(yt, yp),
                macro_f1=f1_score(yt, yp, average="macro", zero_division=0),
                suic_r=recall_score(yt, yp, average=None, zero_division=0)[S])

z = np.load(PREDS); mask = np.load(MASK)
labels, clean = z["labels"], ~mask["strict"]
y = labels[clean]
preds = {k: z[k][clean] for k in z.files if k != "labels"}
n = len(y)
print(f"PART 1: paired bootstrap, B={B:,}, n={n:,} (leakage-filtered)\n")

rng = np.random.default_rng(SEED)
boot = {k: {m: np.empty(B) for m in ("acc", "macro_f1", "suic_r")} for k in preds}
for b in range(B):
    i = rng.integers(0, n, n)          # one resample, shared across all models
    yt = y[i]
    for k, yp in preds.items():
        mv = metrics(yt, yp[i])
        for m in mv:
            boot[k][m][b] = mv[m]

rows = []
for k, yp in preds.items():
    pt = metrics(y, yp); r = {"Model": k.replace("_", " ")}
    for m, lbl in [("acc", "Acc"), ("macro_f1", "MacroF1"), ("suic_r", "Suic-R")]:
        lo, hi = np.percentile(boot[k][m], [2.5, 97.5])
        r[lbl] = round(pt[m], 4)
        r[lbl + " 95% CI"] = f"[{lo:.4f}, {hi:.4f}]"
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

print("\n" + "=" * 78)
print(f"DEPLOYED vs REFERENCE: paired differences")
print("=" * 78)
for m, lbl in [("acc", "accuracy"), ("macro_f1", "macro-F1"), ("suic_r", "Suicidal recall")]:
    d = boot[DEP][m] - boot[REF][m]
    lo, hi = np.percentile(d, [2.5, 97.5])
    verdict = "distinguishable from zero" if (lo > 0 or hi < 0) else "NOT distinguishable from zero"
    print(f"  {lbl:16} {d.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]   {verdict}")

print("\n" + "=" * 78)
print("MLPerf criterion: deployed accuracy as a fraction of the reference")
print("=" * 78)
ratio = boot[DEP]["acc"] / boot[REF]["acc"]
lo, hi = np.percentile(ratio, [2.5, 97.5])
pt = metrics(y, preds[DEP])["acc"] / metrics(y, preds[REF])["acc"]
print(f"  point estimate {100*pt:.2f}%   95% CI [{100*lo:.2f}%, {100*hi:.2f}%]")
print(f"  P(ratio >= {MLPERF:.2f}) = {(ratio >= MLPERF).mean():.3f}")
if lo < MLPERF < hi:
    print(f"  -> the interval straddles the {MLPERF:.0%} threshold: the criterion is met")
    print("     at the point estimate but not robustly, and should be reported that way.")

ratio_r = boot[DEP]["suic_r"] / boot[REF]["suic_r"]
lo2, hi2 = np.percentile(ratio_r, [2.5, 97.5])
print(f"\n  Suicidal-recall ratio {100*ratio_r.mean():.2f}%   95% CI [{100*lo2:.2f}%, {100*hi2:.2f}%]")

pd.DataFrame(rows).to_csv("variance_bootstrap_ci.csv", index=False)
np.savez_compressed("variance_bootstrap_draws.npz",
                    **{f"{k}__{m}": boot[k][m] for k in boot for m in boot[k]})
print("\nWrote variance_bootstrap_ci.csv, variance_bootstrap_draws.npz")

## The app

A small Streamlit front end around the classifier. Writes `app.py` to the working
directory; launch it with `streamlit run app.py` from a terminal in the same folder.

The deployed version differs in two ways: it serves the quantized ONNX model rather
than PyTorch (no torch in the serving image, so the container starts much faster), and
it shows crisis resources whenever P(Suicidal) crosses 0.40 rather than only when
Suicidal is the top label.

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import torch
import pickle

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import kagglehub
from sklearn.preprocessing import LabelEncoder

# Load data to get label mappings (id2label and label2id)
# This part is crucial for correctly interpreting model predictions
suchintikasarkar_sentiment_analysis_for_mental_health_path = kagglehub.dataset_download('suchintikasarkar/sentiment-analysis-for-mental-health')
df_labels_only = pd.read_csv(suchintikasarkar_sentiment_analysis_for_mental_health_path+'/Combined Data.csv', index_col=0)
df_labels_only.dropna(inplace = True)

label_encoder = LabelEncoder()
# Fit on the 'status' column to get all unique labels and create mappings
label_encoder.fit(df_labels_only['status'])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

# Load the BERT tokenizer and model
model_name = "./bert_sentiment_model"
tokenizer = None
bert_model = None
rf_model = None
models_loaded_successfully = False

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id)
    bert_model.eval() # Set to evaluation mode
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    bert_model.to(device)

    # Load the Random Forest model
    # Assuming 'random_forest_bert_model.pkl' is in the same directory as app.py or accessible
    with open('random_forest_bert_model.pkl', 'rb') as file:
        rf_model = pickle.load(file)

    models_loaded_successfully = True
    st.success("Models loaded successfully!")
except Exception as e:
    st.error(f"Error loading models: {e}. Classification functionality will be disabled.")
    models_loaded_successfully = False

# Function to get BERT embeddings for a single text input
def get_bert_embedding_for_text(text, tokenizer, bert_model, device):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        outputs = bert_model.bert(**inputs) # Access the base BERT model to get the last hidden state
        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return cls_embedding

st.title("Mental Health Sentiment Analysis App")
st.subheader("Classify Text Input using BERT + Random Forest")

text_input = st.text_input("Enter text here to classify its sentiment:")

if text_input and models_loaded_successfully:
    st.write(f"You entered: '{text_input}'")
    st.write("---")

    with st.spinner("Classifying..."):
        try:
            # Get BERT embeddings
            bert_embeddings = get_bert_embedding_for_text(text_input, tokenizer, bert_model, device)

            # Predict with Random Forest
            prediction_label_id = rf_model.predict(bert_embeddings)[0]
            prediction_label = id2label[prediction_label_id]

            st.success(f"Classification Result: **{prediction_label}**")

            # Optionally, show raw prediction probabilities
            prediction_probabilities = rf_model.predict_proba(bert_embeddings)[0]
            st.write("Prediction Probabilities:")
            prob_df = pd.DataFrame({
                'Label': [id2label[i] for i in range(len(id2label))],
                'Probability': prediction_probabilities
            }).sort_values(by='Probability', ascending=False).reset_index(drop=True)
            st.dataframe(prob_df)

        except Exception as e:
            st.error(f"An error occurred during prediction: {e}")
elif text_input and not models_loaded_successfully:
    st.warning("Models could not be loaded, so classification is disabled.")
elif not text_input:
    st.info("Please enter some text in the box above to get a classification.")

## Where this ended up

- The Random Forest head and the frozen encoder both failed, for opposite reasons: one
  didn't reduce cost, the other destroyed accuracy. Together they pointed at replacing
  the encoder, which is what worked.
- Quantized DistilBERT is roughly 4x faster and 6x smaller than the BERT reference on
  CPU, for about 0.8 accuracy points.
- Quantization is not uniform across classes. It's near-lossless on DistilBERT but takes
  a large bite out of the two smallest classes on BERT, which overall accuracy hides
  almost completely.
- The corpus has about 5.9% train/test leakage concentrated in the smallest classes, so
  it inflates macro-F1 far more than accuracy.
- The model handles hyperbole reasonably but not sarcasm or temporal hedging, and when
  it fails it tends to fail toward Normal, the wrong direction for a triage tool.

Worth doing next: control items for the edge-case suite, running the probes through the
fp32 model to see whether compression caused the pragmatic failures or the task is just
hard, and the seed replication in Part 2 above.